# DCGAN và mode collapse

GAN không có loss nào giảm đều để nhìn: hai mạng đấu nhau, loss của cả hai dao động quanh một điểm cân bằng. Vì vậy
kỹ năng quan trọng nhất khi train GAN là **đo**. Bài này có hai phần:

1. **Đồ chơi 2D**: hỗn hợp 8 Gauss trên một vòng tròn, nơi bạn biết đáp án nên đo được chính xác GAN phủ bao nhiêu mode.
   So hàm loss minimax gốc (saturating), non-saturating và WGAN-GP qua nhiều seed.
2. **DCGAN trên MNIST**: dựng đúng các quy tắc kiến trúc của Radford et al. (2016), viết một bước train, rồi chẩn đoán
   mẫu sinh bằng một classifier: mỗi lớp chữ số chiếm bao nhiêu phần, mẫu có rõ không, và D(x), D(G(z)) đi thế nào.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`. Hàm loss và `mode_coverage` là của bài Basic 02, ở đây
  viết lại trực tiếp trong vòng train.
- Cell gắn tag `explore` đọc dữ liệu thật, train, vẽ; test bỏ qua chúng. Cell đầu tiên tải MNIST nếu chưa có.
- Env gốc (CPU): phần 2D khoảng 5 phút, DCGAN 3 cấu hình × 3 epoch × 20 000 ảnh khoảng 8 phút. Kernel `ai-gpu`: 15 epoch × 60 000 ảnh.
- Chấm: `uv run pytest tracks/ai-engineer/modules/06-generative-models/exercises/intermediate/02-dcgan-mode-collapse -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải MNIST nếu chưa có
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MNIST = ROOT / "data/raw/mnist"
if not (MNIST.is_dir() and any(MNIST.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("mnist")
print(MNIST)

In [ ]:
import gzip
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


def load_mnist(root: Path, split: str) -> tuple[torch.Tensor, torch.Tensor]:
    """MNIST idx gz → (X float32 (N, 1, 28, 28) trong [-1, 1], y int64). split: "train" | "t10k"."""
    def read_idx(path: Path) -> np.ndarray:
        data = gzip.open(path).read()
        ndim = data[3]
        shape = np.frombuffer(data, ">u4", ndim, offset=4).astype(int)
        return np.frombuffer(data, np.uint8, offset=4 + 4 * ndim).reshape(shape)

    X = read_idx(Path(root) / f"{split}-images-idx3-ubyte.gz").astype(np.float32) / 127.5 - 1
    y = read_idx(Path(root) / f"{split}-labels-idx1-ubyte.gz").astype(np.int64)
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y)

## 1. Dữ liệu đồ chơi: 8 Gauss trên vòng tròn

`ring_data(n, generator, n_modes=8, radius=2.0, std=0.02)`: mỗi điểm chọn ngẫu nhiên đều một mode $$k$$ trong
`0..n_modes-1`, tâm $$(r\cos\theta_k, r\sin\theta_k)$$ với $$\theta_k = 2\pi k/\text{n\_modes}$$, cộng nhiễu Gauss độ lệch `std`
mỗi trục. Mọi số ngẫu nhiên từ `generator`. Trả tensor `(n, 2)` float32.

`ring_centers(n_modes=8, radius=2.0)`: tensor `(n_modes, 2)` các tâm theo đúng thứ tự $$k$$.

In [ ]:
def ring_centers(n_modes: int = 8, radius: float = 2.0) -> torch.Tensor:
    """Tâm (r cos θ_k, r sin θ_k), θ_k = 2πk / n_modes, shape (n_modes, 2)."""
    # TODO
    raise NotImplementedError


def ring_data(n: int, generator: torch.Generator, n_modes: int = 8, radius: float = 2.0, std: float = 0.02) -> torch.Tensor:
    """n điểm từ hỗn hợp đều n_modes Gauss đẳng hướng độ lệch std, tâm trên vòng bán kính radius."""
    # TODO
    raise NotImplementedError

## 2. DCGAN cho ảnh 28×28

Quy tắc kiến trúc của DCGAN (Radford et al., 2016): không pooling, dùng conv có stride (D) và conv transpose (G);
BatchNorm ở cả hai mạng trừ tầng output của G và tầng input của D; G dùng ReLU (Tanh ở output), D dùng LeakyReLU 0.2;
khởi tạo mọi trọng số conv từ $$\mathcal{N}(0, 0.02^2)$$. Bản thu nhỏ cho MNIST:

| G (`z_dim → 1×28×28`) | D (`1×28×28 → logit`) |
| --- | --- |
| `ConvTranspose2d(z_dim, 4c, 7, 1, 0)` → BN → ReLU: `4c×7×7` | `Conv2d(1, c, 4, 2, 1)` → LeakyReLU: `c×14×14` |
| `ConvTranspose2d(4c, 2c, 4, 2, 1)` → BN → ReLU: `2c×14×14` | `Conv2d(c, 2c, 4, 2, 1)` → BN → LeakyReLU: `2c×7×7` |
| `ConvTranspose2d(2c, 1, 4, 2, 1)` → Tanh: `1×28×28` | `Conv2d(2c, 1, 7, 1, 0)`: logit |

- `Generator(z_dim=64, c=64)`: thuộc tính `net` là `nn.Sequential` đúng thứ tự bảng; conv nào đứng trước BatchNorm thì
  `bias=False` (BN đã có $$\beta$$). `forward(z)` nhận `z` shape `(B, z_dim)` và trả ảnh `(B, 1, 28, 28)` trong `[-1, 1]`.
- `Discriminator(c=64)`: thuộc tính `net`, `bias=False` cho conv đứng trước BN. `forward(x)` trả **logit** shape `(B,)`,
  không sigmoid.
- `dcgan_init(module)`: dùng với `model.apply(dcgan_init)`: `Conv2d`/`ConvTranspose2d` trọng số $$\mathcal{N}(0, 0.02^2)$$, bias
  (nếu có) bằng 0; `BatchNorm2d` trọng số $$\mathcal{N}(1, 0.02^2)$$, bias 0. Module khác không đổi.

In [ ]:
class Generator(nn.Module):
    """DCGAN G cho 28×28: z (B, z_dim) → ảnh (B, 1, 28, 28) trong [-1, 1]."""

    def __init__(self, z_dim: int = 64, c: int = 64):
        # TODO
        raise NotImplementedError

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError


class Discriminator(nn.Module):
    """DCGAN D cho 28×28: ảnh (B, 1, 28, 28) → logit (B,)."""

    def __init__(self, c: int = 64):
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError


def dcgan_init(module: nn.Module) -> None:
    """Conv/ConvTranspose: N(0, 0.02²), bias 0. BatchNorm2d: weight N(1, 0.02²), bias 0."""
    # TODO
    raise NotImplementedError

## 3. Một bước train

`gan_step(G, D, opt_g, opt_d, real, generator, loss="ns")` làm đúng một bước của thuật toán 1 (Goodfellow 2014) với
$$k = 1$$, theo thứ tự:

1. Lấy `z = torch.randn(B, G.z_dim, generator=generator)` (một lần, trên CPU rồi chuyển sang device của `real`),
   `fake = G(z)`.
2. **Bước D**: loss $$-\big[\overline{\log D(x)} + \overline{\log(1 - D(G(z)))}\big]$$ trên `real` và `fake` đã **tách khỏi graph của
   G**; `opt_d.zero_grad()`, backward, `opt_d.step()`.
3. **Bước G** trên chính `fake` đó, cho qua D **sau khi D đã cập nhật**: `loss="ns"` là $$-\overline{\log D(G(z))}$$,
   `loss="saturating"` là $$\overline{\log(1 - D(G(z)))}$$; `opt_g.zero_grad()`, backward, `opt_g.step()`.

Trả dict số thực Python: `d_loss`, `g_loss`, `d_real` (trung bình $$D(x)$$ trên `real`) và `d_fake` (trung bình $$D(G(z))$$
ở bước D, trước khi D cập nhật). Viết loss bằng logits (`softplus`), không qua `sigmoid` rồi `log`.

`class_histogram(pred, n_classes=10, min_frac=0.01)`: từ nhãn dự đoán của các mẫu sinh, trả
`(số lớp có tần suất ≥ min_frac, KL(p̂ ‖ đều) theo nats)`, với $$\hat p$$ là tần suất từng lớp và quy ước $$0\log 0 = 0$$.
KL bằng 0 khi mọi lớp đều nhau, bằng $$\log 10 \approx 2.303$$ khi chỉ ra một lớp.

In [ ]:
def gan_step(G: nn.Module, D: nn.Module, opt_g: torch.optim.Optimizer, opt_d: torch.optim.Optimizer,
             real: torch.Tensor, generator: torch.Generator, loss: str = "ns") -> dict[str, float]:
    """Một bước D rồi một bước G (k = 1). Trả d_loss, g_loss, d_real, d_fake (float)."""
    # TODO
    raise NotImplementedError


def class_histogram(pred: torch.Tensor, n_classes: int = 10, min_frac: float = 0.01) -> tuple[int, float]:
    """(số lớp có tần suất ≥ min_frac, KL(p̂ || đều) nats)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: GAN 2D trên vòng 8 Gauss (độ trải std của dữ liệu thật khoảng 1.41). G, D là MLP 2 tầng ẩn 128; 4 cấu hình × 3 seed,
# 3 000 vòng, batch 256, z 2 chiều; WGAN-GP cập nhật critic 5 lần mỗi vòng như bài báo
def mlp(n_in, n_out, act):
    return nn.Sequential(nn.Linear(n_in, 128), act(), nn.Linear(128, 128), act(), nn.Linear(128, n_out))

CENTERS = ring_centers()

def coverage(x, std=0.02, min_count=10):
    "(số mode phủ, tỉ lệ mẫu chất lượng cao) như mode_coverage của bài Basic 02."
    nearest, idx = torch.cdist(x, CENTERS).min(1)
    good = nearest < 3 * std
    return int((torch.bincount(idx[good], minlength=8) >= min_count).sum()), good.float().mean().item()

def train_ring(seed, mode, g_steps=1, iters=3000, bs=256):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    G2, D2 = mlp(2, 2, nn.ReLU), mlp(2, 1, lambda: nn.LeakyReLU(0.2))
    lr, betas = (1e-4, (0.5, 0.9)) if mode == "wgan-gp" else (1e-3, (0.5, 0.999))
    og, od = torch.optim.Adam(G2.parameters(), lr, betas=betas), torch.optim.Adam(D2.parameters(), lr, betas=betas)
    z_eval, traj = torch.randn(2000, 2, generator=torch.Generator().manual_seed(99)), []
    for it in range(1, iters + 1):
        for _ in range(N_CRITIC if mode == "wgan-gp" else 1):
            real, fake = ring_data(bs, g), G2(torch.randn(bs, 2, generator=g)).detach()
            if mode == "wgan-gp":
                a = torch.rand(bs, 1, generator=g)
                xh = (a * real + (1 - a) * fake).requires_grad_()
                (grad,) = torch.autograd.grad(D2(xh).sum(), xh, create_graph=True)
                ld = D2(fake).mean() - D2(real).mean() + GP * ((grad.norm(dim=1) - 1) ** 2).mean()
            else:
                ld = F.softplus(-D2(real).squeeze(1)).mean() + F.softplus(D2(fake).squeeze(1)).mean()
            od.zero_grad(); ld.backward(); od.step()
        for _ in range(g_steps):
            out = D2(G2(torch.randn(bs, 2, generator=g))).squeeze(1)
            lg = {"ns": F.softplus(-out).mean(), "saturating": -F.softplus(out).mean(), "wgan-gp": -out.mean()}[mode]
            og.zero_grad(); lg.backward(); og.step()
        if it % 500 == 0:
            with torch.no_grad():
                traj.append(coverage(G2(z_eval))[0])
    with torch.no_grad():
        return G2(z_eval), traj

N_CRITIC, GP = 5, 0.1
CONFIGS = [("saturating", 1), ("ns", 1), ("ns, G 5 bước/vòng", 5), ("wgan-gp", 1)]
ring_samples, t0 = {}, time.perf_counter()
for name, g_steps in CONFIGS:
    for seed in range(3):
        x, traj = train_ring(seed, name.split(",")[0], g_steps)
        ring_samples[(name, seed)] = x
        modes, hq = coverage(x)
        print(f"{name:18s} seed {seed}: phủ {modes}/8 mode, chất lượng cao {hq:.3f}, độ trải std {x.std(0).mean():.3f}"
              f" | số mode phủ mỗi 500 vòng {traj}", flush=True)
print(f"{time.perf_counter() - t0:.0f} s")

In [ ]:
# explore: mẫu của seed 0 mỗi cấu hình (chấm xám) chồng lên dữ liệu thật (chấm đỏ)
import matplotlib.pyplot as plt

real = ring_data(2000, torch.Generator().manual_seed(5))
fig, axes = plt.subplots(1, len(CONFIGS), figsize=(14, 3.6))
for ax, (name, _) in zip(axes, CONFIGS):
    x = ring_samples[(name, 0)]
    ax.scatter(x[:, 0], x[:, 1], s=2, c="0.4"); ax.scatter(real[:, 0], real[:, 1], s=2, c="tab:red")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_title(name, fontsize=9)
plt.show()

In [ ]:
# explore: MNIST trong [-1, 1]; cấu hình theo device. Classifier CNN nhỏ (2 epoch) làm giám khảo cho mẫu sinh
X_train, y_train = load_mnist(MNIST, "train")
X_test, y_test = load_mnist(MNIST, "t10k")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, N_IMG = (15, 60000) if DEVICE == "cuda" else (3, 20000)
torch.manual_seed(0)
clf = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
                    nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64 * 49, 128), nn.ReLU(), nn.Linear(128, 10)).to(DEVICE)
opt = torch.optim.Adam(clf.parameters(), 1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(2):
    perm = torch.randperm(len(X_train), generator=g)
    for i in range(0, len(X_train), 128):
        b = perm[i:i + 128]
        loss = F.cross_entropy(clf(X_train[b].to(DEVICE)), y_train[b].to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
clf.eval()

@torch.no_grad()
def judge(imgs):
    "(tỉ lệ tự tin > 0.9, số lớp ≥ 1%, KL(p̂ || đều), histogram) trên ảnh trong [-1, 1]."
    p = torch.cat([clf(imgs[i:i + 1000].to(DEVICE)).softmax(1).cpu() for i in range(0, len(imgs), 1000)])
    n_cls, kl = class_histogram(p.argmax(1))
    return round((p.max(1).values > 0.9).float().mean().item(), 3), n_cls, round(kl, 3), torch.bincount(p.argmax(1), minlength=10).tolist()

with torch.no_grad():
    acc = (torch.cat([clf(X_test[i:i + 1000].to(DEVICE)).argmax(1).cpu() for i in range(0, 10000, 1000)]) == y_test).float().mean().item()
print(DEVICE, "| số epoch DCGAN:", EPOCHS, "| ảnh mỗi epoch:", N_IMG, "| classifier test acc:", round(acc, 4), "| giám khảo trên 10 000 ảnh test thật:", judge(X_test)[:3])

In [ ]:
# explore: train DCGAN (Adam lr 2e-4, beta1 0.5, batch 128, z 64 chiều); log theo epoch và chấm 10 000 mẫu sau mỗi epoch
def train_dcgan(epochs, loss="ns", seed=0, lr_d=2e-4):
    torch.manual_seed(seed)
    G, D = Generator().to(DEVICE), Discriminator().to(DEVICE)
    G.apply(dcgan_init); D.apply(dcgan_init)
    opt_g = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr_d, betas=(0.5, 0.999))
    g, z_eval, log = torch.Generator().manual_seed(seed), torch.randn(10000, 64, generator=torch.Generator().manual_seed(123)), []
    for ep in range(epochs):
        t0, stats = time.perf_counter(), []
        G.train(); D.train()
        perm = torch.randperm(len(X_train), generator=g)[:N_IMG]
        for i in range(0, N_IMG - 127, 128):
            real = X_train[perm[i:i + 128]].to(DEVICE)
            stats.append(gan_step(G, D, opt_g, opt_d, real, g, loss))
        G.eval()
        with torch.no_grad():
            samples = torch.cat([G(z_eval[j:j + 1000].to(DEVICE)).cpu() for j in range(0, 10000, 1000)])
        m = {k: float(np.mean([s[k] for s in stats])) for k in stats[0]}
        conf, n_cls, kl, hist = judge(samples)
        log.append({**m, "conf": conf, "classes": n_cls, "kl": kl, "hist": hist})
        print(f"epoch {ep + 1}: d_loss {m['d_loss']:.3f} g_loss {m['g_loss']:.3f} D(x) {m['d_real']:.3f} D(G(z)) {m['d_fake']:.3f}"
              f" | tự tin > 0.9 {conf:.3f}, số lớp {n_cls}, KL {kl:.3f} | {time.perf_counter() - t0:.0f} s", flush=True)
    return G, samples, log

G_ns, samples_ns, log_ns = train_dcgan(EPOCHS)
print("histogram lớp của 10 000 mẫu cuối:", log_ns[-1]["hist"])

In [ ]:
# explore: lưới 8×8 mẫu sinh cuối và đường D(x), D(G(z)) theo epoch
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
grid = samples_ns[:64].reshape(8, 8, 28, 28).permute(0, 2, 1, 3).reshape(8 * 28, 8 * 28)
a1.imshow(grid, cmap="gray"); a1.axis("off"); a1.set_title("DCGAN, non-saturating")
ep = range(1, len(log_ns) + 1)
a2.plot(ep, [r["d_real"] for r in log_ns], marker=".", label="D(x)")
a2.plot(ep, [r["d_fake"] for r in log_ns], marker=".", label="D(G(z))")
a2.plot(ep, [r["conf"] for r in log_ns], marker=".", label="mẫu tự tin > 0.9")
a2.set_xlabel("epoch"); a2.legend()
plt.show()

In [ ]:
# explore: D "quá mạnh" (lr của D gấp 10 lần lr của G), cùng ngân sách: với loss non-saturating và với loss minimax gốc
_, samples_dns, log_dns = train_dcgan(EPOCHS, lr_d=2e-3)
_, samples_dsat, log_dsat = train_dcgan(EPOCHS, loss="saturating", lr_d=2e-3)
for name, lg in [("chuẩn, ns", log_ns), ("D mạnh, ns", log_dns), ("D mạnh, saturating", log_dsat)]:
    r = lg[-1]
    print(f"{name:18s}: tự tin > 0.9 {r['conf']:.3f} | số lớp {r['classes']} | KL {r['kl']:.3f} | D(G(z)) {r['d_fake']:.3f} | histogram {r['hist']}")
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (name, smp) in zip(axes, [("D mạnh, ns", samples_dns), ("D mạnh, saturating", samples_dsat)]):
    ax.imshow(smp[:64].reshape(8, 8, 28, 28).permute(0, 2, 1, 3).reshape(224, 224), cmap="gray"); ax.axis("off"); ax.set_title(name)
plt.show()

## 4. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. Trên vòng 8 Gauss, cấu hình nào phủ đủ 8 mode ở cả 3 seed? Số mode phủ có tăng đều theo thời gian không, hay nhảy lên xuống?
2. Khi G được cập nhật 5 lần mỗi vòng, mẫu sinh trông thế nào (độ trải, số mode)? Giải thích bằng việc G đang tối ưu theo một D "đứng yên".
3. Với DCGAN, D(x) và D(G(z)) có tiến về 0.5 không? Loss của G có giảm đều không, và vì sao đó không phải thước đo chất lượng?
4. Khi D quá mạnh, loss minimax gốc và loss non-saturating khác nhau thế nào? Mode collapse hiện ra ở những con số nào: tỉ lệ tự tin, số
   lớp, KL tới phân phối đều, hay histogram?